In [ ]:
from rag_sql import orchestrate_nl_to_sql, load_all_configs

configs = load_all_configs("config")
out = orchestrate_nl_to_sql("Show clicks and cost by campaign for August 2025", **configs)
out


In [1]:
import os
import sys
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [2]:
# --- Model & runtime knobs ---

MODEL_NAME = "deepseek-r1:8b"   # BEST for SQL agent fallback protocol
EMBED_MODEL_NAME = "llama3:8b"  # embeddings not used yet, safe to leave

BASE_URL = "http://192.168.12.51:11434/v1"   # OpenAI-compatible Ollama endpoint
API_KEY = "test-key"                         # non-empty string required

TOP_K = 6
RETRIES = 2
CTX_CHAR_LIMIT = 8000
ALLOWED_SCHEMA_CHAR_LIMIT = 6000
STRICT_TEMPERATURE = 0
MAX_TOKENS = 800


In [3]:
from rag_sql import load_all_configs

configs = load_all_configs("config")

configs.keys()


dict_keys(['schema', 'metric_registry', 'domain_policy', 'filter_config', 'validator_policy'])

In [4]:
from rag_sql import orchestrate_nl_to_sql


In [5]:
from openai import OpenAI

client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY,
)


In [6]:
SQL_AGENT_SYSTEM = """
You are a SQL Agent for a proprietary advertising data warehouse.

Your job is to decide whether the system can generate SQL.

If the request is clear enough, output:
CALL_SQL: <the user question>

If exactly one critical detail is missing, output:
CLARIFY: <one clarifying question>

Rules:
- Do not generate SQL yourself
- Do not explain your reasoning
- Always output something (CALL_SQL or CLARIFY)
"""


In [7]:
def sql_agent_ask(user_question: str, configs: dict) -> str:
    messages = [
        {"role": "system", "content": SQL_AGENT_SYSTEM.strip()},
        {"role": "user", "content": user_question.strip()},
    ]

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=0.2,
        max_tokens=200,
    )

    text = (resp.choices[0].message.content or "").strip()
    #Debug: see what the LLM actually returned
    print("LLM raw:", repr(text))
    norm = (
        text
        .replace("CALL SQL", "CALL_SQL")
        .replace("CALL_SQL", "CALL_SQL:")
    )

    # Case 1: LLM asks for clarification
    if norm.startswith("CLARIFY:"):
        return norm[len("CLARIFY:"):].strip()


    # Case 2: LLM instructs us to call SQL generator
    if norm.startswith("CALL_SQL:"):
        q2 = norm[len("CALL_SQL:"):].strip()

        out = orchestrate_nl_to_sql(q2, **configs)

        # SQL success
        if out.get("status") == "ok" and out.get("sql"):
            return out["sql"].strip()

        # Clarification needed from orchestrator
        for m in out.get("messages", []):
            if m.get("type") in ("clarify", "needs_clarification"):
                return m.get("detail")

        return "I need one clarification to generate the SQL."

    # Safety fallback: bypass agent
    out = orchestrate_nl_to_sql(user_question, **configs)
    return out.get("sql") or "I need one clarification to proceed."


In [8]:
print("MODEL_NAME:", MODEL_NAME)


MODEL_NAME: deepseek-r1:8b


In [9]:
def sql_agent_ask(user_question: str, configs: dict) -> str:
    out = orchestrate_nl_to_sql(user_question, **configs)

    # Debug (leave on while stabilizing)
    print("ORCH status:", out.get("status"))
    print("ORCH sql present:", bool(out.get("sql")))

    sql = out.get("sql")
    if isinstance(sql, str) and sql.strip():
        return sql.strip()

    # If no SQL, surface the best message
    msgs = out.get("messages") or []
    if msgs:
        # Prefer clarification messages
        for m in msgs:
            if m.get("type") in ("clarify", "needs_clarification", "clarification_question"):
                return m.get("detail") or m.get("question") or "I need one clarification to proceed."
        # Otherwise return first error-ish message
        for m in msgs:
            if m.get("type") in ("validation_error", "planner_error", "validator_error"):
                return m.get("detail") or str(m)
        return msgs[0].get("detail") or str(msgs[0])

    return "I need one clarification to proceed."


In [10]:
print(
    sql_agent_ask(
        "Show clicks and conversions and profit and cost and exchange revenue for Spring Training campaigns on google ads last 7 days",
        configs
    )
)


TypeError: unhashable type: 'list'

In [11]:
print("ORCH status:", out.get("status"))
print("ORCH messages:", out.get("messages"))
print("ORCH sql present:", bool(out.get("sql")))


NameError: name 'out' is not defined

In [23]:
print(
    sql_agent_ask(
        "Show clicks and cost by campaign for August 2025",
        configs
    )
)


SELECT TOP (100) f.[CampaignId] AS [campaign],
       SUM(f.[Clicks]) AS [clicks],
       SUM(f.[Cost]) AS [cost]
FROM [GoogleAdsCampaignPerformanceMetric] AS f
